# Analysis

## Setting parameters

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from quickrules.data_handling import calculate_score, count_all_rules, count_all_attributes, bold
from pathlib import Path
from quickrules.data_handling import balanced_accuracy_score
from sklearn.metrics import accuracy_score, roc_auc_score
import pandas as pd
import numpy as np

In [141]:
data_sets = [  # actual set for QR
    'australian',
     # 'automobile', # cat feats
     'bands',
     'bupa',
     'cleveland',
     # 'contraceptive', # no rules found
     # 'crx', # cat feats
     'dermatology',
     'ecoli',
     # 'german', # cat feats
     'glass',   
     # 'haberman', # no rules found
     'heart',
     'ionosphere',
     # 'mammographic',  # no rules found
     'pima',
     # 'saheart', # cat feats
     'sonar',  # 70 features!
     'spectfheart',
     'vehicle',
     # 'vowel',
     'wine',
     # 'winequality-red',  # todo temp uit voor inclusion tests
     'wisconsin',
     'yeast'
]

def_sets = [
    'australian',
    'bupa',
    'cleveland',
    # 'crx',
    # 'dermatology',
    'ecoli',
    'glass',
    # 'haberman',
    'heart',
    'ionosphere',
    'pima',
    # 'sonar',
    'spectfheart',
    # 'saheart',
    # 'wdbc',
    'wine',
    'winequality-red',
    'wisconsin',
    'yeast',
    # 'automobile',
    # 'dermatology', run base again
    # 'german',
    # 'movement',
    # 'vehicle',
    # 'vowel',
]
inclusion_test = [
    # 'ecoli',
    # 'wisconsin',
    # 'dermatology',
    # 'cleveland',
    # 'spectfheart',
    # 'bupa',
    # 'yeast',
    # 'heart',
    # 'australian',
    # 'glass',
    # 'pima',
    'australian',
     'bands',
     'bupa',
     'cleveland',
     # 'dermatology', temp 01/07/24
     'ecoli',
     'glass',   
     'heart',
     'ionosphere',
     'pima',
     'sonar',  # 70 features! eventjes skip voor QuickReduct 25/04/2024
     'spectfheart',
     'vehicle',
     # 'vowel',
     'wine',
     # 'winequality-red',  # temp uit voor inclusion tests
     'wisconsin',
     'yeast'
]

qr_list = [
    'australian',
    'bupa',
    'cleveland',
    'dermatology',
    'ecoli',
    'glass',
    'heart',
    'pima',
    'spectfheart',
    # 'vehicle',
    # 'vowel',
    'wisconsin',
    'yeast'
]

In [142]:
data_sets = inclusion_test

In [143]:
len(data_sets)

15

In [6]:
data_base = Path.cwd() / 'keel-data'
metric = balanced_accuracy_score  # balanced_
scores = {}
rules = {}
attributes = {}
results_base = Path.cwd() / 'results'

## Loading WEKA results
MODLEM, FURIA, RIPPER

In [7]:
weka_folder = Path.cwd() / 'weka-results'
weka_models = {
    'modlem':'&',
    'furia':'and',
    'ripper':'and'
}
acc_type = 'balanced-accuracy'  # 'weka-accuracy.csv'

In [8]:
for name, connection in weka_models.items():
    model_folder = weka_folder / f"{name}-files"
    file = model_folder / f"{name}-{acc_type}.csv"
    scores[name] = pd.read_csv(weka_folder / file, header=None, index_col=0).to_dict()[1]
    rules[name] = {}
    attributes[name] = {}
    for file in model_folder.iterdir():
        if file.name[-3:] == 'txt':
            short_name = file.name[:-4]
            with open(file, 'r') as f:
                line = f.readline()
                nrs = []
                while len(line) > 4:
                    nrs.append(line.count(connection) + 1)
                    line = f.readline()
            rules[name][short_name] = len(nrs)
            attributes[name][short_name] = np.average(nrs)

## Loading QuickRules results

In [9]:
qr_models = {
    'qr': results_base / 'no-prune-results' / 'close-min-max-combo-results',
    # 'lin-owa': results_base / 'no-prune-results' / 'rules-lin-owa-qr-combo-minmax-results',
    # 'trun-lin-owa': results_base / 'no-prune-results' / 'rules-trun-lin-owa-qr-combo-minmax-results',
    # 'trun-exp-owa': results_base / 'no-prune-results' / 'trun-exp-owa-qr-combo-minmax-results',
    # 'avg-qr': results_base / 'mon-avg-std-minmax-results-2',
    # 'avg-lin-owa' : results_base / 'mon-avg-lin-owa-minmax-results-2',
    # 'prune-qr': results_base / 'prune-results' / 'prune-qr-combo-minmax-results',
    # 'prune-lin-owa': results_base / 'prune-results' / 'prune-lin-owa-qr-combo-minmax-results',
    # 'prune-trun-lin-owa': results_base / 'prune-results' / 'prune-trun-lin-owa-qr-combo-min-max-results',
    # 'prune-trun-exp-owa': results_base / 'prune-results' / 'prune-trun-exp-owa-qr-combo-min-max-results',
    # 'prune-avg-qr': results_base / 'prune-mon-avg-std-minmax-results-2',
    # 'prune-avg-lin-owa' : results_base / 'prune-mon-avg-lin-owa-minmax-results-2',
}

In [10]:
for model, path in qr_models.items():
    scores[model] = calculate_score(
        data_folder=data_base,
        results_folder=path,
        include=data_sets,
        metric=metric
    )
    rules[model] = count_all_rules(
        Path.cwd() / results_base / path,
        include=data_sets
    )
    attributes[model] = count_all_attributes(
        Path.cwd() / results_base / path,
        include=data_sets
    )

## Adding results for non rule based models


In [11]:
other_models =  {
    'naive-bayes': Path.cwd() / 'NaiveBayes-results',
    'svm': Path.cwd() / 'svm',
    'tree': Path.cwd() / 'decision-tree'
}

In [12]:
for model, path in other_models.items():
    print(model)
    scores[model] = calculate_score(
        data_folder=data_base,
        results_folder=path,
        include=data_sets,
        metric=metric,
        verbose=False
    )

naive-bayes
svm
tree


In [13]:
frnn_results = pd.read_csv(Path.cwd() / 'frnn_new_results.csv', header=None, index_col=0).to_dict()[1]
scores['frnn'] =  {data_set: frnn_results[data_set] for data_set in data_sets}

## FRRI Models

In [130]:
frri_models = {
    # 'non-overlap': Path.cwd() / 'non-overlap-rules',
    # 'nori' : Path.cwd() / 'non-overlap-rules-def_sets',
    # 'lower-nori' : Path.cwd() / 'lower-approx-rules-def_sets',
    # 'lower-check' : Path.cwd() / 'lower-approx-og-implement-test',
    # 'lower-new-check' : Path.cwd() / results_base / 'lower-approx-new-impl-test',
    # 'lower-incl-t1': Path.cwd() / results_base / 'lower-approx-luka-impl-incl.99'
    'incl 1e-6' : Path.cwd() / results_base / 'lower-approx' / 'inclusion' / "1-1e-6",
    'incl 1e-4' : Path.cwd() / results_base / 'lower-approx-new-impl-test',
    'incl 1e-3' : Path.cwd() / results_base / 'lower-approx' / 'inclusion' / "999",
    'incl 1e-2' : Path.cwd() / results_base / 'lower-approx' / 'inclusion' / "99",
    # 'incl 0.95' : Path.cwd() / results_base / 'lower-approx' / 'inclusion' / "95",
    'owa 1e-6' : Path.cwd() / results_base / 'lower-approx' / 'linear-owa-inclusion' / '1-1e-6',
    # 'quickreduct 1e-6': Path.cwd() / results_base / 'lower-approx' / 'quickreduct-ordering-bug' / '1-1e-6',
    'cv-thres': Path.cwd() / results_base / 'lower-approx' / 'cv-inclusion',
    'frfs': Path.cwd() / results_base / 'lower-approx' / 'frfs',
    'msecvx-no-reducts': Path.cwd() / results_base / 'multiclassMSECVX' / 'no-reducts',
    'maecvx-no-reducts': Path.cwd() / results_base / 'multiclassMAECVX' / 'no-reducts',
    'msecvx-0.5' : Path.cwd() / 'results' / 'multiclassMSECVX' / 'no-reducts-0.5-threshold',
    'maecvx-0.5-reducts' : Path.cwd() / 'results' / 'multiclassMAECVX' / 'reducts-0.5-threshold',
    'msecvx-0.5-reducts': Path.cwd() / 'results' / 'multiclassMSECVX' / 'reducts-0.5-threshold',
    'msecvx-i95-reducts': Path.cwd() / 'results' / 'multiclassMSECVX' / 'reducts-incl95-cov05',
    'msecvx-i05-reducts': Path.cwd() / 'results' / 'multiclassMSECVX' / 'reducts-incl5-cov05',
    'base-no-reducts': Path.cwd() / results_base / 'inclusion' / 'no-reducts',
    'owa-lower-base': Path.cwd() / results_base / 'owa-lower' / 'default',
    'default-check': Path.cwd() / 'results' / 'default'
}

In [131]:
for model, path in frri_models.items():
    print(model)
    scores[model] = calculate_score(
        data_folder=Path.cwd() / 'keel-data',
        results_folder=path, #'rule_prune-rel-0dot00-std',  #  'rel-0dot05-std', #' no-prune-results'/ 'close-min-max-combo-results',  # /
        metric=metric,
        include=data_sets,
        label_encoding=True
    )
    rules[model] = count_all_rules(
        path,
        include=data_sets
    )
    attributes[model] = count_all_attributes(
        path,
        include=data_sets,
        counter='attribute'
    )

incl 1e-6
incl 1e-4
incl 1e-3
incl 1e-2
owa 1e-6
cv-thres
frfs
msecvx-no-reducts
maecvx-no-reducts
msecvx-0.5
maecvx-0.5-reducts
msecvx-0.5-reducts
msecvx-i95-reducts
msecvx-i05-reducts
base-no-reducts
owa-lower-base
default-check


## Checking all of the models

In [132]:
scores.keys()

dict_keys(['modlem', 'furia', 'ripper', 'qr', 'naive-bayes', 'svm', 'tree', 'frnn', 'incl 1e-6', 'incl 1e-4', 'incl 1e-3', 'incl 1e-2', 'owa 1e-6', 'cv-thres', 'frfs', 'msecvx-no-reducts', 'maecvx-no-reducts', 'base-no-reducts', 'owa-lower-base', 'default-check', 'msecvx-0.5', 'msecvx-0.5-reducts', 'maecvx-0.5-reducts', 'msecvx-i95-reducts', 'msecvx-i05-reducts'])

## Tables
set 1 = QR + OWA-variants without pruning
set 2 = focus on pruning

In [177]:
names = {
    1: [
        'qr',
        'lin-owa',
        'trun-lin-owa',
        'trun-exp-owa',
        'modlem'
    ],
    2: [
        'qr',
        'lin-owa',
        'prune-qr',
        'prune-lin-owa'
    ],
    3: [
        'qr',
        'lin-owa',
        'avg-qr',
        'avg-lin-owa'
    ],
    4: [
        'lin-owa',
        'modlem'
    ],
    5: [
        'lin-owa',
        'frnn',
        'svm',
        'naive-bayes',
        'tree',
    ],
    'frri-paper': [
        # 'nori',
        # 'lower-nori',
        # 'lower-check',
        'lower-new-check',
        # 'lower-incl-t1',
        'modlem',
        'qr',
        'furia',
        'ripper',
    ],
    'inclusion' : [
        'incl 1e-6',
        'incl 1e-4',
        'incl 1e-3', 
        'incl 1e-2', 
        # 'incl 0.95',
        # 'owa 1e-6',
        # 'quickreduct 1e-6',
        'cv-thres',
        # 'frfs'
    ],
    'quickreduct' : [
        'incl 1e-6',
        'quickreduct 1e-6',
        'frfs'
    ],
    'gran' : [
        'base-no-reducts',
        'msecvx-no-reducts',
        'maecvx-no-reducts',
        'msecvx-0.5'
    ],
    'gran-reducts': [
        'incl 1e-4',
        # 'msecvx-0.5-reducts',
        # 'maecvx-0.5-reducts',
        # 'msecvx-i95-reducts',
        'msecvx-i05-reducts'
    ],
    'owa' : [
        'incl 1e-4',
        'owa-lower-base'
    ],
    'default-check': [ # -> OK
        'incl 1e-4',
        'default-check'
    ]
}

In [178]:
nr = 'inclusion' # 6

In [179]:
main_folder = nr # + "small"  # 'tables_inclusion_set2' # 'balanced_acc_incl'  # 'normal_acc'
tables_path_base = Path.cwd() / 'tables' / main_folder

In [180]:
table_acc = pd.DataFrame(index=data_sets, columns=names[nr])

In [181]:
for model in names[nr]:
    for data_set in data_sets:
    # for data_set, score in scores[model].items():
        table_acc.loc[data_set, model] = scores[model][data_set]

In [182]:
# table_acc['max'] = table_acc.max(axis='columns', skipna=True)

In [183]:
table_acc.loc['mean'] = table_acc.mean(axis='rows', skipna=True)

In [184]:
table_acc

,incl 1e-6,incl 1e-4,incl 1e-3,incl 1e-2,cv-thres
australian,0.824936,0.787517,0.813182,0.82039,0.812028
bands,0.664654,0.675141,0.676793,0.60893,0.658848
bupa,0.593125,0.600724,0.594187,0.640177,0.589394
cleveland,0.317538,0.290397,0.314681,0.283667,0.293392
ecoli,0.700635,0.720659,0.69504,0.681103,0.682324
glass,0.668254,0.679067,0.641468,0.497123,0.668552
heart,0.730833,0.7675,0.734167,0.75,0.769167
ionosphere,0.910721,0.912468,0.91078,0.889655,0.901234
pima,0.680919,0.687633,0.673859,0.660827,0.657748
sonar,0.776288,0.786212,0.721313,0.771187,0.720051


In [162]:
bolded_first = table_acc.apply(lambda data: bold(data), axis=1)
bolded_first.to_latex(buf= tables_path_base / f'tab{nr}acc.txt', escape=False)

/var/folders/_f/yf3dkr5x1397v4ntwhncxfhw0000gn/T/ipykernel_53128/3326315922.py:2: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  bolded_first.to_latex(buf= tables_path_base / f'tab{nr}acc.txt', escape=False)


In [163]:
table_rule = pd.DataFrame(index=data_sets, columns=names[nr])
for model in names[nr]:
    # for data_set, rule_count in rules[model].items():
    for data_set in data_sets:
        table_rule.loc[data_set, model] = rules[model][data_set]  #  rule_count
table_rule.loc['mean'] = table_rule.mean(axis='rows', skipna=True)

In [164]:
table_rule

,incl 1e-4,msecvx-i05-reducts
australian,92.3,80.6
bands,59.0,84.7
bupa,76.7,59.7
cleveland,102.2,102.8
ecoli,56.8,62.2
glass,51.1,60.0
heart,44.7,37.9
ionosphere,19.3,26.8
pima,125.0,95.5
sonar,17.8,28.0


In [165]:
bolded_first = table_rule.apply(lambda data: bold(data, optimum='min', format_string="%.0f"), axis=1)
bolded_first.to_latex(buf= tables_path_base / f'tab{nr}rules.txt', escape=False)

/var/folders/_f/yf3dkr5x1397v4ntwhncxfhw0000gn/T/ipykernel_53128/3026783303.py:2: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  bolded_first.to_latex(buf= tables_path_base / f'tab{nr}rules.txt', escape=False)


In [166]:
table_attribute = pd.DataFrame(index=data_sets, columns=names[nr])
for model in names[nr]:
    # for data_set, attribute_count in attributes[model].items():
    for data_set in data_sets:
        table_attribute.loc[data_set, model] = attributes[model][data_set]  # attribute_count
table_attribute.loc['mean'] = table_attribute.mean(axis='rows', skipna=True)

In [167]:
table_attribute

,incl 1e-4,msecvx-i05-reducts
australian,5.154734,7.138659
bands,6.292805,5.270798
bupa,4.202797,4.312907
cleveland,5.449101,4.82389
ecoli,3.854646,3.662146
glass,3.884301,3.599892
heart,4.904402,4.227588
ionosphere,4.527456,8.510326
pima,5.080997,4.702172
sonar,7.798125,5.855025


In [168]:
bolded_first = table_attribute.apply(lambda data: bold(data, optimum='min', format_string="%.2f"), axis=1)
bolded_first.to_latex(buf= tables_path_base / f'tab{nr}atts.txt', escape=False)

/var/folders/_f/yf3dkr5x1397v4ntwhncxfhw0000gn/T/ipykernel_53128/3212901870.py:2: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  bolded_first.to_latex(buf= tables_path_base / f'tab{nr}atts.txt', escape=False)


## Statistical testing

In [170]:
from scipy import stats
import scikit_posthocs as sph

In [185]:
subject = table_acc

In [186]:
no_mean =  subject.drop(labels = 'mean', axis = 'index')
# no_mean.drop('max', axis='columns', inplace=True)

In [187]:
no_mean

,incl 1e-6,incl 1e-4,incl 1e-3,incl 1e-2,cv-thres
australian,0.824936,0.787517,0.813182,0.82039,0.812028
bands,0.664654,0.675141,0.676793,0.60893,0.658848
bupa,0.593125,0.600724,0.594187,0.640177,0.589394
cleveland,0.317538,0.290397,0.314681,0.283667,0.293392
ecoli,0.700635,0.720659,0.69504,0.681103,0.682324
glass,0.668254,0.679067,0.641468,0.497123,0.668552
heart,0.730833,0.7675,0.734167,0.75,0.769167
ionosphere,0.910721,0.912468,0.91078,0.889655,0.901234
pima,0.680919,0.687633,0.673859,0.660827,0.657748
sonar,0.776288,0.786212,0.721313,0.771187,0.720051


In [208]:
stats.wilcoxon(no_mean['cv-thres'],no_mean['incl 1e-6'], alternative='two-sided')

/Users/henri/Library/CloudStorage/OneDrive-Personal/Documents/_Work/PhD Thesis/2022-fuzzylem/venv/lib/python3.10/site-packages/scipy/stats/_morestats.py:3414: UserWarning: Exact p-value calculation does not work if there are zeros. Switching to normal approximation.
  warnings.warn("Exact p-value calculation does not work if there are "


WilcoxonResult(statistic=19.0, pvalue=0.03546470751403722)

In [128]:
f_test = stats.friedmanchisquare(*no_mean.values)
f_test

FriedmanchisquareResult(statistic=72.60000000000002, pvalue=1.5315170910874449e-09)

In [129]:
post_hocs = sph.posthoc_conover_friedman(no_mean.values, p_adjust='Holm')
post_hocs.columns=no_mean.columns
post_hocs.index=no_mean.columns
post_hocs

,incl 1e-6,incl 1e-4,incl 1e-3,incl 1e-2,cv-thres
incl 1e-6,1.000000,1.0,1.0,0.757743,0.858925
incl 1e-4,1.000000,1.0,1.0,1.000000,1.000000
incl 1e-3,1.000000,1.0,1.0,1.000000,1.000000
incl 1e-2,0.757743,1.0,1.0,1.000000,1.000000
cv-thres,0.858925,1.0,1.0,1.000000,1.000000


In [65]:
best = no_mean.max(axis='columns')
dif = no_mean.subtract(best, axis='rows').abs()
dif.max(axis='rows').sort_values()

incl 1e-3     0.04599
incl 1e-6    0.047052
cv-thres     0.050783
incl 1e-4    0.050873
incl 1e-2    0.181944
dtype: object

In [189]:
ranks = no_mean.rank(axis='columns', ascending=False)
friedman_ranks = ranks.mean()
print(friedman_ranks.sort_values().to_latex(escape=False))

\begin{tabular}{lr}
\toprule
{} &         0 \\
\midrule
incl 1e-6 &  2.433333 \\
incl 1e-4 &  2.600000 \\
incl 1e-3 &  2.900000 \\
cv-thres  &  3.433333 \\
incl 1e-2 &  3.633333 \\
\bottomrule
\end{tabular}


/var/folders/_f/yf3dkr5x1397v4ntwhncxfhw0000gn/T/ipykernel_53128/2662355144.py:3: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  print(friedman_ranks.sort_values().to_latex(escape=False))


In [67]:
ranks['lower-new-check'].value_counts()

KeyError: 'lower-new-check'

In [68]:
ranks.apply(pd.value_counts)

,incl 1e-6,incl 1e-4,incl 1e-3,incl 1e-2,cv-thres
1.0,3,5.0,1.0,3,2
2.0,4,3.0,3.0,1,2
2.5,1,NaN,1.0,1,1
3.0,5,1.0,7.0,1,2
4.0,1,1.0,3.0,3,5
5.0,1,5.0,NaN,6,3


In [188]:
median = no_mean.median(axis='rows')
median

incl 1e-6    0.680919
incl 1e-4    0.687633
incl 1e-3    0.676793
incl 1e-2    0.660827
cv-thres     0.668552
dtype: float64

On high IR datasets (>6)

In [196]:
ir_datasets = [
    'cleveland',
    'ecoli',
    'glass',
    'spectfheart',
    'winequality-red',
    'yeast'
]

ir_no_mean = no_mean.loc[ir_datasets]
# ir_no_mean.drop('qr', axis='columns', inplace=True)

In [197]:
f_test = stats.friedmanchisquare(*ir_no_mean.values)
f_test

FriedmanchisquareResult(statistic=14.714285714285708, pvalue=0.011655524770004939)

In [198]:
post_hocs = sph.posthoc_conover_friedman(ir_no_mean.values)
post_hocs.columns=ir_no_mean.columns
post_hocs.index=ir_no_mean.columns
post_hocs

,lower-new-check,modlem,qr,furia,ripper
lower-new-check,1.000000,0.233052,0.004900,0.094249,0.094249
modlem,0.233052,1.000000,0.067585,0.603959,0.603959
qr,0.004900,0.067585,1.000000,0.175230,0.175230
furia,0.094249,0.603959,0.175230,1.000000,1.000000
ripper,0.094249,0.603959,0.175230,1.000000,1.000000


In [199]:
stats.wilcoxon(ir_no_mean['lower-new-check'], ir_no_mean['ripper'], alternative="greater")

WilcoxonResult(statistic=18.0, pvalue=0.078125)

In [200]:
wil_ph = sph.posthoc_nemenyi_friedman(ir_no_mean.values)
wil_ph.columns=ir_no_mean.columns
wil_ph.index=ir_no_mean.columns
wil_ph

,lower-new-check,modlem,qr,furia,ripper
lower-new-check,1.000000,0.680037,0.008987,0.359170,0.359170
modlem,0.680037,1.000000,0.261866,0.900000,0.900000
qr,0.008987,0.261866,1.000000,0.576422,0.576422
furia,0.359170,0.900000,0.576422,1.000000,0.900000
ripper,0.359170,0.900000,0.576422,0.900000,1.000000
